# Learning Curves — Spotting Overfitting

**Train loss ↓** but **val loss ↑** → model memorizes training data.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Simulate train vs val metrics

In [ ]:
epochs = np.arange(1, 21)
# Synthetic curves mimicking overfitting after epoch 12
train_loss = 1.0 * np.exp(-epochs / 8) + 0.05 + np.random.randn(20) * 0.02
val_loss = 1.0 * np.exp(-epochs / 10) + 0.1 + np.maximum(0, (epochs - 12) * 0.03) + np.random.randn(20) * 0.02
train_acc = 1 - train_loss / 1.2
val_acc = 1 - val_loss / 1.3

## 2. Train vs val loss

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, train_loss, 'b-o', label='train loss', lw=2)
ax.plot(epochs, val_loss, 'r-o', label='val loss', lw=2)
ax.axvline(12, color='gray', ls='--', label='overfitting starts')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend()
ax.set_title('Train vs validation loss')
plt.tight_layout(); plt.show()

## 3. Train vs val accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, train_acc, 'b-o', label='train acc', lw=2)
ax.plot(epochs, val_acc, 'r-o', label='val acc', lw=2)
ax.axvline(12, color='gray', ls='--')
ax.set_xlabel('epoch'); ax.set_ylabel('accuracy'); ax.legend()
ax.set_title('Accuracy curves — gap widens when overfitting')
plt.tight_layout(); plt.show()

## 4. Real mini-training on synthetic data

In [ ]:
gen = DummyDataGenerator(n_samples=300, n_features=8, n_classes=3)
X, y = gen.tensors()
n = len(y); tr, va = X[:200], X[200:]; ytr, yva = y[:200], y[200:]
model = SimpleMLP()
opt = torch.optim.Adam(model.parameters(), lr=0.02)
tl, vl = [], []
for ep in range(30):
    model.train()
    opt.zero_grad()
    loss = F.cross_entropy(model(tr), ytr)
    loss.backward(); opt.step()
    tl.append(loss.item())
    model.eval()
    with torch.no_grad():
        vl.append(F.cross_entropy(model(va), yva).item())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tl, label='train'); ax.plot(vl, label='val')
ax.legend(); ax.set_title('Live training on DummyDataGenerator')
plt.tight_layout(); plt.show()

## 5. Generalization gap

In [ ]:
gap = np.array(vl) - np.array(tl)
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(range(len(gap)), 0, gap, alpha=0.4, color='orange')
ax.plot(gap, color='darkorange', lw=2)
ax.set_xlabel('epoch'); ax.set_ylabel('val_loss - train_loss')
ax.set_title('Generalization gap (wider = more overfitting)')
plt.tight_layout(); plt.show()